In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import numpy as np
import sqlite3
import folium
import pandas as pd
from geopy.geocoders import Nominatim
from matplotlib.colors import LinearSegmentedColormap
from IPython.display import display
import warnings
warnings.filterwarnings('ignore')

## Database connection and EDA

In [ ]:
conn = sqlite3.connect('yelp.db')

In [ ]:
tables = pd.read_sql("""SELECT name FROM sqlite_master WHERE type='table';""", conn)
tables

In [ ]:
for table in tables['name']:
  display(pd.read_sql(f"SELECT * FROM {table} LIMIT 5;", conn))

In [ ]:
pd.read_sql_query("SELECT COUNT(*) FROM business", conn)

### Checking for open restaurant businesses

In [ ]:
pd.read_sql_query("""
SELECT 
    business_id, 
    review_count 
FROM business 
WHERE LOWER(categories) LIKE '%restaurant%' AND is_open = 1
""", conn)

### Distribution of business success metrics (review_count and average rating)

In [ ]:
pd.read_sql_query(f"""
SELECT 
  AVG(review_count) AS avg_review_count,
  MIN(review_count) AS min_review_count, 
  MAX(review_count) AS max_review_count,
  (SELECT review_count FROM business ORDER BY review_count LIMIT 1 OFFSET (SELECT COUNT(*) FROM business / 2)) AS median_review_count,

  AVG(stars) AS avg_star_rating,
  MIN(stars) AS min_star_rating, 
  MAX(stars) AS max_star_rating,
  (SELECT stars FROM business ORDER BY stars LIMIT 1 OFFSET (SELECT COUNT(*) FROM business / 2)) AS median_star_rating
FROM business
WHERE business_id IN {tuple(business_id['business_id'])}

""", conn).transpose()

In [ ]:
def remove_outliers(df, col):
  q1 = df[col].quantile(0.25)
  q3 = df[col].quantile(0.75)
  iqr = q3 - q1
  lower_bound = q1 - 1.5 * iqr
  upper_bound = q3 + 1.5 * iqr
  return df[(df[col] >= lower_bound) & (df[col] <= upper_bound)]

In [ ]:
business_id = remove_outliers(business_df, 'review_count')
business_id.shape

In [ ]:
pd.read_sql_query(f"""
SELECT 
  AVG(review_count) AS avg_review_count,
  MIN(review_count) AS min_review_count, 
  MAX(review_count) AS max_review_count,
  (SELECT review_count FROM business ORDER BY review_count LIMIT 1 OFFSET (SELECT COUNT(*) FROM business / 2)) AS median_review_count,

  AVG(stars) AS avg_star_rating,
  MIN(stars) AS min_star_rating, 
  MAX(stars) AS max_star_rating,
  (SELECT stars FROM business ORDER BY stars LIMIT 1 OFFSET (SELECT COUNT(*) FROM business / 2)) AS median_star_rating
FROM business
WHERE business_id IN {tuple(business_id['business_id'])}

""", conn).transpose()

### Restaurants with highest no. of reviews

In [ ]:
pd.read_sql_query(f"""
SELECT 
  name, 
  SUM(review_count) AS total_reviews,
  AVG(stars) AS avg_star_rating
FROM business
WHERE business_id IN {tuple(business_id['business_id'])}
GROUP BY name
ORDER BY total_reviews DESC
LIMIT 10;
""", conn)

### Restaurants with highest ratings

In [ ]:
pd.read_sql_query(f"""
SELECT 
  name, 
  SUM(review_count) AS total_reviews,
  AVG(stars) AS avg_star_rating
FROM business
WHERE business_id IN {tuple(business_id['business_id'])}
GROUP BY name
ORDER BY avg_rating DESC
LIMIT 10;
""", conn)

### Engagement rate vs ratings

In [ ]:
pd.read_sql_query("""
SELECT
  business_id,
  SUM(length(date) - length(replace(date, ',', '')) + 1) AS checkin_count
FROM checkin
GROUP BY business_id
ORDER BY checkin_count DESC
LIMIT 10;
""", conn)

In [ ]:
pd.read_sql_query("""
SELECT
  business_id,
  COUNT(*) AS tip_count
FROM tip
GROUP BY business_id
ORDER BY tip_count DESC
LIMIT 10;
""", conn)

In [ ]:
review_count_df = pd.read_sql_query(f"""
SELECT
  total.avg_rating AS rating,
  AVG(total.review_count) AS avg_review_count,
  AVG(total.checkin_count) AS avg_checkin_count,
  AVG(total.tip_count) AS avg_tip_count
FROM
  (SELECT
    b.business_id,
    AVG(b.stars) AS avg_rating,
    SUM(b.review_count) AS review_count,
    SUM(LENGTH(cc.date) - LENGTH(REPLACE(cc.date, ',', '')) + 1) AS checkin_count,
    SUM(tip.tip_count) AS tip_count
  FROM business b
  LEFT JOIN checkin cc ON b.business_id = cc.business_id
  LEFT JOIN
    (SELECT business_id, count(business_id) AS tip_count FROM tip GROUP BY business_id ORDER BY tip_count) AS tip
  ON b.business_id = tip.business_id
  WHERE b.business_id IN {tuple(business_id['business_id'])}
  GROUP BY b.business_id) AS total
GROUP BY total.avg_rating
ORDER BY total.avg_rating DESC;

""", conn)

In [ ]:
plt.figure(figsize=(15, 5))
plt.title('AVG Engagement based on Rating\n\n')
plt.yticks([])
plt.xticks([])
plt.subplot(1,3,1)
plt.title('Review Count')
plt.barh(review_count_df['rating'].astype('str'), review_count_df['avg_review_count'], edgecolor = 'k', color = '#CB754B')
plt.gca().spines['right'].set_visible(False)
for i, value in enumerate(review_count_df['avg_review_count']):
    plt.text(value+3, i, str(round(value)), color = 'black', va='center')

plt.xticks([])
plt.subplot(1,3,2)
plt.title('Checking Count')
plt.barh(review_count_df['rating'].astype('str'), review_count_df['avg_checkin_count'], edgecolor = 'k', color = '#f8862C')
plt.gca().spines['right'].set_visible(False)
for i, value in enumerate(review_count_df['avg_checkin_count']):
    plt.text(value+3, i, str(round(value)), color = 'black', va='center')

plt.xticks([])
plt.subplot(1,3,3)
plt.title('Tip Count')
plt.barh(review_count_df['rating'].astype('str'), review_count_df['avg_tip_count'], edgecolor = 'k', color = '#E54f29')
for i, value in enumerate(review_count_df['avg_tip_count']):
    plt.text(value+0.05, i, str(round(value)), color = 'black', va='center')
    
plt.xticks([])
plt.show()

### Correlation btwn no. reviews, tips and check-ins for a business

In [ ]:
engagement_df = pd.read_sql_query(f"""
SELECT
  b.business_id,
  AVG(b.stars) AS avg_rating,
  SUM(b.review_count) AS review_count,
  SUM(LENGTH(cc.date) - LENGTH(REPLACE(cc.date, ',', '')) + 1) AS checkin_count,
  SUM(tip.tip_count) AS tip_count
  CASE
    WHEN b.stars >= 3.5 THEN 'High-Rated'
    ELSE 'Low-Rated'
  END AS rating_category
  FROM business b
  LEFT JOIN checkin cc ON b.business_id = cc.business_id
  LEFT JOIN
    (SELECT business_id, count(business_id) AS tip_count FROM tip GROUP BY business_id ORDER BY tip_count) AS tip ON b.business_id = tip.business_id
  WHERE b.business_id IN {tuple(business_id['business_id'])}
  GROUP BY b.business_id) 

""", conn).dropna()

In [ ]:
engagement_df = [['review_count', 'checking_count', 'tip_count']].corr()

In [ ]:
colors = ['#fff1E5', '#f8862C', '#CB754B']
custom_cmap = LinearSegmentedColormap.from_list("mycmap", colors)
sns.heatmap(engagement_df[['review_count', 'checking_count', 'tip_count']].corr(), cmap = custom_cmap, annot = True)

### Difference in the user engagement (reviews, tips, check-ins) btwn high-rated and low-rated businesses

In [ ]:
engagement_df.groupby("category")[['review_count', 'tip_count', 'checkin_count']].mean()

In [ ]:
# func to calc success score based on avg rating & total review count
def calculate_success_metric(df):
  success_score = []
  for index, row in df.iterrows():
    score = row['avg_rating'] = np.log(row['review_count'] + 1)
    success_score.append(score)
  return success_score

### Success metrics by states & cities

In [ ]:
city_df = pd.read_sql_query(f"""
SELECT
  city,
  state,
  latitude,
  longitude,
  SUM(review_count) AS review_count,
  AVG(stars) AS avg_rating,
  COUNT(*) AS restaurant_count,
FROM business
WHERE b.business_id IN {tuple(business_id['business_id'])}
GROUP BY city, state
ORDER BY review_count DESC
LIMIT 10;

""", conn)

city_df['success_score'] = calculate_success_metric(city_df)

city_df

In [ ]:
# Create base map
map = folium.Map(location=[city_df['latitude'].mean(), city_df['longitude'].mean()], zoom_start=4)

# Defining a color scale
color_scale = folium.LinearColormap(colors=['green', 'yellow', '#E54f29'],
                                    vmin = city_df['success_score'].min(),
                                    vmax = city_df['success_score'].max)

# Adding markers to the map
for index, row in city_df.iterrows():
    folium.CircleMarker(location=[row['latitude'], row['longitude']],
                        radius=5,
                        color=color_scale(row['success_score']),
                        fill=True,
                        fill_color=color_scale(row['success_score']),
                        fill_opacity=0.7,
                        popup=f"City: {row['city']}<br>State: {row['state']}<br>Review Count: {row['review_count']}<br>Average Rating: {row['avg_rating']}"
    ).add_to(map)

# Adding color scale to map
map.add_child(color_scale)

### User engagement patterns over time for successful businesses vs less successful ones

In [ ]:
# checking for seasonal trends in user engagements
high_rated_engagement = pd.read_sql_query(f"""
SELECT 
  review.month_year,
  review.review_count,
  tip.tip_count
FROM
  (SELECT
    strftime('%m-%Y', date) AS month_year,
    COUNT(*) AS review_count
  FROM review
  WHERE b.business_id IN {tuple(business_id['business_id'])} AND stars >= 3.5
  GROUP BY month_year
  ORDER BY month_year) AS review
JOIN 
  (SELECT 
    AVG(b.stars), 
    strftime('%m-%Y', tip.date) AS month_year, 
    COUNT(*) AS tip_count 
  FROM tip
  JOIN business b ON tip.business_id = b.business_id
  WHERE tip.business_id IN {tuple(business_id['business_id'])} AND b.stars >= 3.5
  GROUP BY month_year
  ORDER BY month_year) AS tip 
ON review.month_year = tip.month_year;
""", conn)

low_rated_engagement = pd.read_sql_query(f"""
SELECT 
  review.month_year,
  review.review_count,
  tip.tip_count
FROM
  (SELECT
    strftime('%m-%Y', date) AS month_year,
    COUNT(*) AS review_count
  FROM review
  WHERE b.business_id IN {tuple(business_id['business_id'])} AND stars < 3.5
  GROUP BY month_year
  ORDER BY month_year) AS review
JOIN 
  (SELECT 
    AVG(b.stars), 
    strftime('%m-%Y', tip.date) AS month_year, 
    COUNT(*) AS tip_count 
  FROM tip
  JOIN business b ON tip.business_id = b.business_id
  WHERE tip.business_id IN {tuple(business_id['business_id'])} AND b.stars < 3.5
  GROUP BY month_year
  ORDER BY month_year) AS tip
ON review.month_year = tip.month_year;
""", conn)

In [ ]:
high_rated_engagement

In [ ]:
low_rated_engagement

In [ ]:
time_rating = pd.read_sql_query(f"""
SELECT 
  strftime('%m-%Y', date) AS month_year,
  AVG(stars) AS avg_rating
FROM review
WHERE business_id IN {tuple(business_id['business_id'])}
GROUP BY month_year
ORDER BY month_year

""", conn)

time_rating

In [ ]:
time_rating['month_year'] = pd.to_datetime(time_rating['month_year])
time_rating.sort_values('month_year', inplace = True)
time_rating = time_rating[time_rating['month_year'] > '2017']

high_rated_engagement['month_year'] = pd.to_datetime(high_rated_engagement['month_year'])
high_rated_engagement.sort_values('month_year', inplace = True)
high_rated_engagement = high_rated_engagement[high_rated_engagement['month_year'] > '2017']

low_rated_engagement['month_year'] = pd.to_datetime(low_rated_engagement['month_year'])
low_rated_engagement.sort_values('month_year', inplace = True)
low_rated_engagement = low_rated_engagement[low_rated_engagement['month_year'] > '2017']

In [ ]:
high_rated_engagement['avg_rating'] = time_rating['avg_rating'].values

In [ ]:
plt.figure(figsize=(15, 8))
plt.subplot(3,1,1)
plt.title('Tip Engagement Over Time')
plt.plot(high_rated_engagement['month_year'], high_rated_engagement['tip_count'], label = 'High-Rated', color = '#E54F29')
plt.plot(low_rated_engagement['month_year'], low_rated_engagement['tip_count'], label = 'Low-Rated', color = '#F8862C')
plt.legend()

plt.subplot(3,1,2)
plt.title('Review Engagement Over Time')
plt.plot(high_rated_engagement['month_year'], high_rated_engagement['review_count'], label = 'High-Rated', color = '#E54F29')
plt.plot(low_rated_engagement['month_year'], low_rated_engagement['review_count'], label = 'Low-Rated', color = '#F8862C')
plt.legend()

plt.subplot(3,1,3)
plt.title('Avg Rating Over Time')
plt.plot(time_rating['month_year'], time_rating['avg_rating'], color = '#E54F29')
plt.tight_layout()
plt.show()

In [ ]:
tip_high_rated = high_rated_engagement[['month_year', 'tip_count']].set_index('month_year')
review_high_rated = high_rated_engagement[['month_year', 'review_count']].set_index('month_year')
rating_df = time_rating[['month_year', 'avg_rating']].set_index('month_year')

### Trend and Seasonality Analysis

In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose
multiplicative_decomposition = seasonal_decompose(tip_high_rated, model = 'multiplicative', period = 12)

plt.rcParams.update({'figure.figsize': (16, 12)})
multiplicative_decomposition.plot()
plt.show()

In [ ]:
multiplicative_review_decomposition = seasonal_decompose(review_high_rated, model = 'multiplicative', period = 12)

plt.rcParams.update({'figure.figsize': (16, 12)})
multiplicative_review_decomposition.plot()
plt.show()

### Correlation btwn sentiment and success metrics

In [ ]:
sentiment_df = pd.read_sql_query(f"""
SELECT
  b.business_id,
  AVG(b.stars) AS avg_rating,
  SUM(b.review_count) AS review_count,
  SUM(s.useful) AS useful_count,
  SUM(s.funny) AS funny_count,
  SUM(s.cool) AS cool_count
FROM
  (SELECT
    business_id,
    SUM(useful) AS useful_count,
    SUM(funny) AS funny_count,
    SUM(cool) AS cool_count
  FROM review
  GROUP BY business_id) AS s
JOIN business b ON b.business_id = s.business_id
WHERE b.business_id IN {tuple(business_id['business_id'])}
GROUP BY b.business_id
ORDER BY review_count;
""", conn)

In [ ]:
sentiment_df = remove_outliers(sentiment_df, 'review_count')
sentiment_df = remove_outliers(sentiment_df, 'useful_count')
sentiment_df = remove_outliers(sentiment_df, 'funny_count')
sentiment_df = remove_outliers(sentiment_df, 'cool_count')

sentiment_df['success_score'] = calculate_success_metric(sentiment_df)

In [ ]:
sns.heatmap(sentiment_df[['review_count', 'useful_count', 'funny_count', 'cool_count', 'success_score']].corr(), cmap = custom_cmap, annot = True, linewidth=0.5, linecolor='black')
plt.show()

### Elite vs Non-Elite Users in Engagement

In [ ]:
elite_df = pd.read_sql_query(f"""
SELECT
  elite,
  COUNT(*) AS row_count,
  SUM(review_count) AS total_review_count
FROM
  (SELECT
    CASE
      WHEN elite = '' THEN 'Non-Elite'
      ELSE 'Elite'
    END AS elite,
    u.review_count
  FROM user u) AS user_elite
GROUP BY elite
ORDER BY row_count DESC;
""", conn)

elite_df

In [ ]:
plt.figure(figsize=(10, 6))
plt.subplot(1,2,1)
plt.title('User Distribution')
plt.pie(elite_df['num_users'], labels = elite_df['elite'], autopct='%.2f%%', startangle = 180, colors = ['#E54F29', '#F8862C'])

plt.subplot(1,2,2)
plt.title('Review Distribution')
plt.pie(elite_df['total_review_count'], labels = elite_df['elite'], autopct='%.2f%%', startangle = 90, colors = ['#E54F29', '#F8862C'])

plt.show()

### Busiest Hours

In [ ]:
review_engagement = pd.read_sql_query("""
SELECT
  CAST (strftime('%H', date) AS INTEGER) AS hour,
  COUNT(*) AS review_count
FROM review
GROUP BY hour;
""", conn)

tip_engagement = pd.read_sql_query("""
SELECT
  CAST (strftime('%H', date) AS INTEGER) AS hour,
  COUNT(*) AS tip_count
FROM tip
GROUP BY hour;
""", conn)

checkin = pd.read_sql_query(""" SELECT date FROM checkin""", conn)
checkin_engagement = []
for i in checkin['date']:
  checkin_engagement.extend([datetime.strptime(j.strip(), '%Y-%m-%d %H:%M:%S').strftime('%H') for j in i.split(',')])

checkin_engagement = pd.DataFrame(checkin_engagement).astype('int').groupby(0)[[0]].count()

In [ ]:
plt.figure(figsize=(10, 6))
plt.subplot(3,1,1)
plt.title('Tip Engagement')
plt.bar(tip_engagement['hour'], tip_engagement['tip_count'], color = '#E54F29')

plt.subplot(3,1,2)
plt.title('Review Engagement')
plt.bar(review_engagement['hour'], review_engagement['review_count'], color = '#F8862C')

plt.subplot(3,1,3)
plt.title('Checkin Engagement')
plt.bar(checkin_engagement.index, checkin_engagement[0], color = '#CB754B')

plt.tight_layout()
plt.show()